In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_json("data/llm_hallucination_detection_dataset.jsonl", lines=True)
unique = df.drop_duplicates(["prompt", "context", "response"]).copy()
train_df, test_df = train_test_split(unique, test_size=0.2, random_state=42)
print(len(train_df), len(test_df))

153 39


In [2]:
import re

def overlap(response, context):
    response_words = set(re.findall(r"[a-z0-9]+", str(response).lower()))
    context_words = set(re.findall(r"[a-z0-9]+", str(context).lower()))
    if not response_words:
        return 0
    return len(response_words & context_words) / len(response_words)

word_pred = test_df.apply(
    lambda row: "hallucinated" if overlap(row["response"], row["context"]) < 0.6 else "faithful",
    axis=1,
)
print((word_pred == test_df["label"]).mean())

0.9743589743589743


In [3]:
import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

label2id = {"faithful": 0, "hallucinated": 1}
id2label = {0: "faithful", 1: "hallucinated"}

class HallucinationDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=256):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        hypothesis = str(row["prompt"]) + " " + str(row["response"])
        encoded = self.tokenizer(
            str(row["context"]),
            hypothesis,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item["labels"] = torch.tensor(label2id[row["label"]])
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {"accuracy": accuracy_score(labels, preds), "weighted_f1": f1}

def train_and_score(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, id2label=id2label, label2id=label2id
    )
    args = TrainingArguments(
        output_dir=f"outputs/{model_name}",
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        eval_strategy="no",
        save_strategy="no",
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=HallucinationDataset(train_df, tokenizer),
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    predicted = trainer.predict(HallucinationDataset(test_df, tokenizer))
    y_pred = [id2label[i] for i in predicted.predictions.argmax(axis=-1)]
    print(model_name)
    print(classification_report(test_df["label"], y_pred, digits=4))

C:\Users\Sh_al\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
train_and_score("bert-base-uncased")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 323.93it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

Step,Training Loss


bert-base-uncased
              precision    recall  f1-score   support

    faithful     1.0000    0.8000    0.8889        20
hallucinated     0.8261    1.0000    0.9048        19

    accuracy                         0.8974        39
   macro avg     0.9130    0.9000    0.8968        39
weighted avg     0.9153    0.8974    0.8966        39



In [5]:
train_and_score("roberta-base")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 334.64it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\Sh_al\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memor

Step,Training Loss


roberta-base
              precision    recall  f1-score   support

    faithful     0.9412    0.8000    0.8649        20
hallucinated     0.8182    0.9474    0.8780        19

    accuracy                         0.8718        39
   macro avg     0.8797    0.8737    0.8715        39
weighted avg     0.8813    0.8718    0.8713        39



## Result on the 39 unique test rows

Word check: accuracy 0.974.
BERT-base: accuracy 0.872.
RoBERTa-base: accuracy 0.872.

Both models miss the same number of hallucinated answers as the word check, and they also mark some faithful answers as hallucinated. The word check stays the current method.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

text_train = train_df["prompt"] + " " + train_df["response"]
text_test = test_df["prompt"] + " " + test_df["response"]

model = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
model.fit(text_train, train_df["label"])
print(classification_report(test_df["label"], model.predict(text_test), digits=4))

              precision    recall  f1-score   support

    faithful     0.9375    0.7500    0.8333        20
hallucinated     0.7826    0.9474    0.8571        19

    accuracy                         0.8462        39
   macro avg     0.8601    0.8487    0.8452        39
weighted avg     0.8620    0.8462    0.8449        39



## No context

A logistic regression model trained on the question and the answer only, with the context left out, scores 0.846 accuracy on the 39 test rows. The word check scores 0.974 and needs the context. BERT and RoBERTa with the context score 0.872.

In [7]:
class NoContextDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=256):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        text = str(row["prompt"]) + " " + str(row["response"])
        encoded = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item["labels"] = torch.tensor(label2id[row["label"]])
        return item

def train_and_score_no_context(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, id2label=id2label, label2id=label2id
    )
    args = TrainingArguments(
        output_dir=f"outputs/nocontext-{model_name}",
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        eval_strategy="no",
        save_strategy="no",
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=NoContextDataset(train_df, tokenizer),
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    predicted = trainer.predict(NoContextDataset(test_df, tokenizer))
    y_pred = [id2label[i] for i in predicted.predictions.argmax(axis=-1)]
    print(model_name, "no context")
    print(classification_report(test_df["label"], y_pred, digits=4))

In [8]:
train_and_score_no_context("bert-base-uncased")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 616.03it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

Step,Training Loss


bert-base-uncased no context
              precision    recall  f1-score   support

    faithful     0.5455    0.6000    0.5714        20
hallucinated     0.5294    0.4737    0.5000        19

    accuracy                         0.5385        39
   macro avg     0.5374    0.5368    0.5357        39
weighted avg     0.5376    0.5385    0.5366        39



In [9]:
train_and_score_no_context("roberta-base")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 413.12it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\Sh_al\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memor

Step,Training Loss


roberta-base no context
              precision    recall  f1-score   support

    faithful     1.0000    0.2500    0.4000        20
hallucinated     0.5588    1.0000    0.7170        19

    accuracy                         0.6154        39
   macro avg     0.7794    0.6250    0.5585        39
weighted avg     0.7851    0.6154    0.5544        39



## BERT and RoBERTa with no context

Input is the question and the answer only. The context column is not used.

RoBERTa accuracy: 0.615.
Word check with context: 0.974.
BERT and RoBERTa with context: 0.872.
Logistic regression with no context: 0.846.
